In [1]:
import scanpy as sc
import Spectra as spc
import pandas as pd
import numpy as np
from Spectra import Spectra_util as spc_tl
from Spectra import K_est as kst
from Spectra import default_gene_sets

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/Spectra/Spectra_util.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
annotations = spc.default_gene_sets.load()

In [3]:
# Load your data
adata = sc.read_h5ad('patient_data/anal_pc5_c21_S1.filtered.h5ad')

In [4]:
metadata = pd.read_csv('data/anal_cancer_n21_meta.csv', low_memory=False)
metadata.head()

,Unnamed: 0,samplename,viral_status,cluster_annot,copykat,pathology_status,cluster_annot2,cluster_annot3
0,S1_AAACAAGCAACAGCACACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,T_cells,not.defined,cancer,CD4+Tcells,Tregs
1,S1_AAACAAGCAATTGAGTACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Myeloid_cells,diploid,cancer,DC_CLEC9A,DC_CLEC9A
2,S1_AAACAAGCACGTAAATACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Fibroblast_stromal,not.defined,cancer,CD8+Tcells,Fibroblasts/Stromal
3,S1_AAACAAGCACTATCACACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,B_cells,not.defined,cancer,Plasma_XBP1,Plasma_XBP1
4,S1_AAACAAGCAGAATGAAACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Endothelials,diploid,cancer,Endothelials,Endothelials


In [5]:
metadata = metadata.set_index('Unnamed: 0')
adata.obs['cluster_annot3'] = metadata.loc[adata.obs_names, 'cluster_annot3']

In [6]:
print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 14133 × 17781
    obs: 'orig.ident', 'cluster_annot3'
    obsm: 'X_harmony', 'X_pca', 'X_umap'
                                           orig.ident       cluster_annot3
S1_AAACAAGCAACAGCACACTTTAGG_anal_34449_1_1         S1                Tregs
S1_AAACAAGCAATTGAGTACTTTAGG_anal_34449_1_1         S1            DC_CLEC9A
S1_AAACAAGCACGTAAATACTTTAGG_anal_34449_1_1         S1  Fibroblasts/Stromal
S1_AAACAAGCACTATCACACTTTAGG_anal_34449_1_1         S1          Plasma_XBP1
S1_AAACAAGCAGAATGAAACTTTAGG_anal_34449_1_1         S1         Endothelials


In [7]:
# Check what cell types are in the default annotations
print("Default annotation keys:", annotations.keys())

Default annotation keys: dict_keys(['B_GC', 'B_memory', 'B_naive', 'CD4_T', 'CD8_T', 'DC', 'ILC3', 'MDC', 'NK', 'Treg', 'gdT', 'mast', 'pDC', 'plasma', 'global'])


In [8]:
# Check what cell types are in the data
print("Cell types in your data:")
print(metadata.cluster_annot3.value_counts())

Cell types in your data:
cluster_annot3
Tumor/Epithelials      171927
Fibroblasts/Stromal     33617
Plasma_XBP1             30276
Endothelials            13279
Mac_C1Q_ZEB2            11520
CD8+Tcells               8802
CD4+Tcells               7574
Tregs                    4528
Bcells_CD19              3986
Mac_C1Q_SPP1             2586
NKcells                  2509
Mon_VCAN                 2489
Mast                     2463
DC_CD207                 2376
CD8+Tcells_Ki67          1868
Neutrophils              1818
DC_LAMP3                 1412
DC_CLEC9A                 838
pDCs                      766
Name: count, dtype: int64


In [9]:
# Map cell types to Spectra's expected cell type labels
annotation_mapping = {
    # T cells
    'CD4+Tcells': 'CD4_T',
    'CD8+Tcells': 'CD8_T',
    'CD8+Tcells_Ki67': 'CD8_T',  # proliferating CD8s still map to CD8_T
    'Tregs': 'Treg',
    'NKcells': 'NK',
    
    # B cells and plasma
    'Bcells_CD19': 'B_naive',  # or B_memory - depends on your data
    'Plasma_XBP1': 'plasma',
    
    # Dendritic cells
    'DC_CD207': 'DC',      # Langerhans/cDC2
    'DC_LAMP3': 'DC',      # mature/migratory DCs
    'DC_CLEC9A': 'DC',     # cDC1
    'pDCs': 'pDC',
    
    # Myeloid
    'Mac_C1Q_ZEB2': 'MDC',  # resident macrophages
    'Mac_C1Q_SPP1': 'MDC',  # inflammatory/SPP1+ macrophages
    'Mon_VCAN': 'MDC',      # classical monocytes
    'Neutrophils': 'MDC',   # if you need to include them
    
    # Other immune
    'Mast': 'mast',
    
    # Non-immune cell types - map to custom labels (not 'global'!)
    'Tumor/Epithelials': 'Epithelial',
    'Fibroblasts/Stromal': 'Fibroblast',
    'Endothelials': 'Endothelial',
}

In [10]:
# Apply the mapping to create the spectra_annot_celltype column
metadata['spectra_annot_celltype'] = metadata['cluster_annot3'].map(annotation_mapping)
# Add to adata
adata.obs['spectra_annot_celltype'] = metadata.loc[adata.obs_names, 'spectra_annot_celltype']

# Check for any unmapped cell types (will be NaN)
print("Unique mapped cell types in adata:")
print(adata.obs['spectra_annot_celltype'].value_counts(dropna=False))

Unique mapped cell types in adata:
spectra_annot_celltype
Epithelial     7299
Fibroblast     1621
plasma         1329
B_naive        1326
CD4_T           624
CD8_T           551
Endothelial     414
MDC             335
Treg            300
DC              136
NK               97
mast             92
pDC               9
Name: count, dtype: int64


In [11]:
# Build annotations_custom with ONLY the cell types present in the data

# Get the unique cell types actually in data (excluding NaN)
adata_celltypes = set(adata.obs['spectra_annot_celltype'].dropna().unique())
print("Cell types in your adata:", adata_celltypes)

# only include 'global' and cell types that exist in the data
annotations_custom = {'global': annotations['global']}
for ct in adata_celltypes:
    if ct in annotations:
        # This cell type has gene sets in the default annotations
        annotations_custom[ct] = annotations[ct]
    else:
        # This cell type (Epithelial, Fibroblast, Endothelial) has no gene sets
        # Add an empty dict so Spectra knows about it
        annotations_custom[ct] = {}

print("\nAnnotations_custom keys:", annotations_custom.keys())

Cell types in your adata: {'Fibroblast', 'CD8_T', 'CD4_T', 'pDC', 'B_naive', 'Endothelial', 'plasma', 'Treg', 'NK', 'Epithelial', 'MDC', 'mast', 'DC'}

Annotations_custom keys: dict_keys(['global', 'Fibroblast', 'CD8_T', 'CD4_T', 'pDC', 'B_naive', 'Endothelial', 'plasma', 'Treg', 'NK', 'Epithelial', 'MDC', 'mast', 'DC'])


In [12]:
# Check if already normalized (values should be small, typically 0-10 range)
print("Max value before normalization:", adata.X.max())

# If max is large, data is raw counts - normalize it
if adata.X.max() > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Max value after normalization:", adata.X.max())
else:
    print("Data appears to already be normalized")

Max value before normalization: 1552
Max value after normalization: 9.023591


In [13]:
annotations_filtered = spc_tl.check_gene_set_dictionary(
    adata,
    annotations_custom,
    obs_key='spectra_annot_celltype',
    global_key='global'
)
print("Gene set dictionary check passed!")

Cell type labels in gene set annotation dictionary and AnnData object are identical
Your gene set annotation dictionary is now correctly formatted.
Gene set dictionary check passed!


In [14]:
# Compute highly variable genes 
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

Number of highly variable genes: 3000


In [15]:
# FIX: Monkey-patch pandas Series to add nonzero() method
# This fixes compatibility between newer pandas and scipy sparse indexing
# The issue is that Spectra's code creates boolean Series that scipy can't handle

if not hasattr(pd.Series, 'nonzero'):
    def _series_nonzero(self):
        return self.to_numpy().nonzero()
    pd.Series.nonzero = _series_nonzero
    print("Applied pandas Series.nonzero() patch")

Applied pandas Series.nonzero() patch


In [16]:
# Fit the Spectra model
model = spc.est_spectra(
    adata=adata, 
    gene_set_dictionary=annotations_filtered,  # use the filtered annotations
    use_highly_variable=True,
    cell_type_key="spectra_annot_celltype", 
    use_weights=True,
    lam=0.1,  # varies depending on data and gene sets, try between 0.5 and 0.001
    delta=0.001, 
    kappa=None,
    rho=0.001, 
    use_cell_types=True,
    n_top_vals=50,
    label_factors=True, 
    overlap_threshold=0.2,
    clean_gs=True, 
    min_gs_num=3,
    num_epochs=500  # use 10000 for real runs
)

Cell type labels in gene set annotation dictionary and AnnData object are identical
Your gene set annotation dictionary is now correctly formatted.


100%|██████████| 500/500 [10:45<00:00,  1.29s/it]


In [17]:
# View results stored in adata
print("Cell scores shape:", adata.obsm['SPECTRA_cell_scores'].shape)
print("\nFactors stored in adata.uns['SPECTRA_factors']")
print("Markers stored in adata.uns['SPECTRA_markers']")

Cell scores shape: (14133, 191)

Factors stored in adata.uns['SPECTRA_factors']
Markers stored in adata.uns['SPECTRA_markers']


interpreting spectra ouputs

In [18]:
print(adata.uns['SPECTRA_L'])

{'global': 151, 'Fibroblast': 1, 'CD8_T': 7, 'CD4_T': 12, 'pDC': 2, 'B_naive': 1, 'Endothelial': 1, 'plasma': 1, 'Treg': 2, 'NK': 1, 'Epithelial': 1, 'MDC': 6, 'mast': 2, 'DC': 3}


In [ ]:
# === USAGE MATRIX (Cells × Factors) ===
# Get factor labels for nice column names
factor_labels = adata.uns['SPECTRA_overlap'].index.tolist()

usage_matrix = pd.DataFrame(
    adata.obsm['SPECTRA_cell_scores'],
    index=adata.obs_names,
    columns=factor_labels
)
print(f"Usage matrix shape: {usage_matrix.shape}")

# === GEP MATRIX (Factors × Genes) ===
# Get the vocabulary (genes used in model)
vocab_genes = adata.var[adata.var['spectra_vocab']].index.tolist()

gep_matrix = pd.DataFrame(
    adata.uns['SPECTRA_factors'],
    index=factor_labels,
    columns=vocab_genes
)
print(f"GEP matrix shape: {gep_matrix.shape}")

Usage matrix shape: (14133, 191)
GEP matrix shape: (191, 6520)


In [20]:
usage_matrix.head()

,0-X-global-X-all_biotin_metabolism,1-X-global-X-all_DNA-repair,2-X-global-X-leuko_ROS_production,3-X-global-X-all_amino-sugar-nucleotide-sugar_metabolism,4-X-global-X-all_cholesterol-homeostasis,5-X-global-X-all_carnitine-shuttle,6-X-global-X-all_TLR_signaling,7-X-global-X-all_phosphoinositide_signaling,8-X-global-X-all_macroautophagy,9-X-global-X-all_TCA-cycle,...,181-X-MDC-X-181,182-X-MDC-X-182,183-X-NK-X-TNK_cytotoxicity-effectors,184-X-Treg-X-Treg_FoxP3-stabilization,185-X-Treg-X-185,186-X-mast-X-all_multidrug-resistance,187-X-mast-X-187,188-X-pDC-X-p-DC_CpG-TLR9_response,189-X-pDC-X-189,190-X-plasma-X-all_transmembrane-transport-ER
S1_AAACAAGCAACAGCACACTTTAGG_anal_34449_1_1,1.591368e-09,1.744080e-08,8.796550e-11,1.025783e-07,9.158491e-04,3.999212e-10,2.560915e-03,1.926363e-03,1.251294e-09,1.996032e-07,...,0.0,0.0,0.0,0.024476,0.039352,0.0,0.0,0.0,0.0,0.00000
S1_AAACAAGCAATTGAGTACTTTAGG_anal_34449_1_1,2.485473e-08,2.387290e-08,7.215633e-11,8.800211e-08,2.560400e-04,4.565757e-09,8.993199e-09,6.752323e-08,3.805667e-09,2.721203e-07,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000
S1_AAACAAGCACGTAAATACTTTAGG_anal_34449_1_1,1.637756e-10,4.442934e-04,4.670347e-10,5.414966e-08,3.377726e-04,1.856018e-08,4.688173e-08,2.074155e-07,6.766537e-10,3.054226e-03,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000
S1_AAACAAGCACTATCACACTTTAGG_anal_34449_1_1,1.051657e-10,1.743026e-08,4.382522e-11,9.770647e-09,2.514607e-07,3.270519e-07,7.758192e-06,2.974182e-08,1.190233e-08,5.566554e-07,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.02059
S1_AAACAAGCAGAATGAAACTTTAGG_anal_34449_1_1,2.589693e-09,3.344811e-04,9.668130e-10,6.031587e-04,1.668867e-09,1.405269e-09,7.232685e-08,1.361571e-04,1.827676e-09,3.531923e-09,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000


In [24]:
gep_matrix.head()

,SAMD11,ISG15,TNFRSF18,TNFRSF4,B3GALT6,UBE2J2,TAS1R3,DVL1,MXRA8,NADK,...,RAB41,SLC7A3,ADGRG4,PCDH11Y,AMELY,KLHL40,IL31,ALAS2,ATP1B4,B3GAT2
0-X-global-X-all_biotin_metabolism,1.861280e-19,2.040290e-12,8.238436e-12,4.865969e-17,2.108600e-05,5.651326e-06,2.562564e-18,7.024747e-06,1.631172e-08,1.338429e-16,...,6.646179e-17,5.756090e-19,6.342915e-16,1.107816e-17,6.581213e-17,4.825426e-16,1.211512e-16,2.743484e-19,9.957581e-17,2.410895e-16
1-X-global-X-all_DNA-repair,1.729360e-19,4.876477e-14,5.659255e-15,3.347290e-17,3.768825e-14,5.814397e-14,2.300382e-18,5.177831e-16,2.853950e-14,1.294564e-18,...,4.095826e-17,7.608552e-19,6.812812e-16,8.627877e-18,5.176878e-17,1.713704e-16,9.434927e-17,1.735648e-19,1.831136e-17,1.074744e-17
2-X-global-X-leuko_ROS_production,2.297006e-19,1.769544e-15,5.877056e-11,4.039328e-17,4.641791e-08,2.578888e-18,3.050419e-18,2.335125e-21,1.120797e-18,3.155866e-21,...,3.082306e-17,2.421287e-19,3.223225e-16,8.078225e-18,4.920731e-17,1.219801e-16,9.833252e-17,2.472534e-20,4.019610e-19,9.033175e-19
3-X-global-X-all_amino-sugar-nucleotide-sugar_metabolism,1.543573e-19,8.992630e-14,6.952480e-12,3.155314e-17,1.339101e-13,1.410925e-10,2.771670e-18,2.863767e-11,4.284679e-14,4.677423e-17,...,3.394575e-17,9.795510e-19,3.332894e-16,9.422989e-18,5.411029e-17,1.485191e-15,1.298680e-16,1.824921e-19,1.081762e-16,1.149403e-16
4-X-global-X-all_cholesterol-homeostasis,2.458225e-19,8.986372e-14,5.093866e-15,8.507592e-17,1.650829e-15,3.446691e-14,4.726547e-18,4.989951e-15,9.878576e-15,9.932098e-18,...,9.588512e-17,5.137943e-19,8.156326e-16,1.039126e-17,6.347042e-17,6.264297e-16,5.928047e-17,1.007487e-19,1.875040e-16,7.855101e-17


In [29]:
print(adata.uns['SPECTRA_markers'][0])  # top genes for factor 0

['HLCS' 'SLC5A6' 'BTD' 'SLC19A3' 'AKR7A2' 'DDX10' 'PIDD1' 'CETN2' 'BMPR1A'
 'COX5B' 'DFFA' 'BID' 'PSMG1' 'MPC2' 'COL1A2' 'PLOD2' 'DESI1' 'UROS'
 'PNO1' 'SUPT4H1' 'ETFDH' 'GNG12' 'AP2A2' 'SKP2' 'TUBGCP5' 'HSPE1' 'ALG8'
 'NTHL1' 'PIK3CA' 'SEMA3B' 'XPOT' 'SASS6' 'XRCC6' 'FOXRED2' 'EIF4E' 'LSM1'
 'TIMM9' 'NDUFB1' 'BPHL' 'COX7B' 'HSPA4L' 'CYCS' 'TOMM40' 'RSL1D1' 'PSMB3'
 'ATP5F1C' 'ACVR1' 'ACSM3' 'RNF14' 'HSPBP1']
